# Environment Test: Session-based XLNet with Transformers4Rec

Self-contained notebook that generates synthetic data and runs the full XLNet training pipeline.
Use this to validate that the Merlin/Transformers4Rec stack works on your VM.

In [ ]:
import os
import glob
import shutil
import math

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch

from transformers4rec import torch as tr
from transformers4rec.torch.ranking_metric import NDCGAt, AvgPrecisionAt, RecallAt
from transformers4rec.torch.utils.examples_utils import wipe_memory

## Generate Synthetic Data

Generate fake session sequences and write them as parquet files in the directory structure expected by the trainer.

In [ ]:
INPUT_DATA_DIR = os.environ.get("INPUT_DATA_DIR", "/workspace/data")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", f"{INPUT_DATA_DIR}/sessions_by_day")

NUM_ITEMS = 500
NUM_CATEGORIES = 50
MAX_SEQ_LEN = 20
TRAIN_ROWS = 1000
EVAL_ROWS = 200
NUM_DAYS = 8  # days 1-7 for train, 2-8 for eval

np.random.seed(42)


def generate_sessions(n_rows, max_seq_len=MAX_SEQ_LEN):
    """Generate synthetic session data matching NVTabular output format."""
    import pandas as pd

    item_ids = []
    categories = []
    weekday_sins = []
    age_days = []

    for _ in range(n_rows):
        seq_len = np.random.randint(2, max_seq_len + 1)
        item_ids.append(np.random.randint(1, NUM_ITEMS + 1, size=seq_len).tolist())
        categories.append(np.random.randint(1, NUM_CATEGORIES + 1, size=seq_len).tolist())
        weekday_sins.append(np.sin(np.random.uniform(0, 2 * math.pi, size=seq_len)).tolist())
        age_days.append(np.random.uniform(0, 365, size=seq_len).tolist())

    df = pd.DataFrame({
        "item_id-list": item_ids,
        "category-list": categories,
        "weekday_sin-list": weekday_sins,
        "age_days-list": age_days,
    })
    return df


# Create directory structure and write parquet files
os.makedirs(os.path.join(INPUT_DATA_DIR, "processed_nvt"), exist_ok=True)

# Write schema-source parquet
schema_df = generate_sessions(TRAIN_ROWS)
schema_df.to_parquet(os.path.join(INPUT_DATA_DIR, "processed_nvt/part_0.parquet"))

# Write daily train/eval parquet files
for day in range(1, NUM_DAYS + 1):
    day_dir = os.path.join(OUTPUT_DIR, str(day))
    os.makedirs(day_dir, exist_ok=True)
    generate_sessions(TRAIN_ROWS).to_parquet(os.path.join(day_dir, "train.parquet"))
    generate_sessions(EVAL_ROWS).to_parquet(os.path.join(day_dir, "valid.parquet"))

print(f"Synthetic data written to {INPUT_DATA_DIR}")
print(f"Daily sessions written to {OUTPUT_DIR} (days 1-{NUM_DAYS})")

## Load Schema & Select Features

In [ ]:
from merlin.schema import Schema, Tags, ColumnSchema
from merlin.io import Dataset

train_ds = Dataset(os.path.join(INPUT_DATA_DIR, "processed_nvt/part_0.parquet"))
schema = train_ds.schema

# Tag features so Transformers4Rec knows how to handle them
schema["item_id-list"] = schema["item_id-list"].with_tags([Tags.CATEGORICAL, Tags.ITEM_ID, Tags.ITEM, Tags.LIST, Tags.SEQUENCE])
schema["category-list"] = schema["category-list"].with_tags([Tags.CATEGORICAL, Tags.LIST, Tags.SEQUENCE])
schema["weekday_sin-list"] = schema["weekday_sin-list"].with_tags([Tags.CONTINUOUS, Tags.LIST, Tags.SEQUENCE])
schema["age_days-list"] = schema["age_days-list"].with_tags([Tags.CONTINUOUS, Tags.LIST, Tags.SEQUENCE])

# Set cardinalities for categorical features
schema["item_id-list"] = schema["item_id-list"].with_properties({"domain": {"min": 0, "max": 500, "name": "item_id"}, "embedding_sizes": {"cardinality": 501, "dimension": 64}})
schema["category-list"] = schema["category-list"].with_properties({"domain": {"min": 0, "max": 50, "name": "category"}, "embedding_sizes": {"cardinality": 51, "dimension": 16}})

schema = schema.select_by_name([
    'item_id-list',
    'category-list',
    'weekday_sin-list',
    'age_days-list',
])

schema

## Define Model

In [ ]:
inputs = tr.TabularSequenceFeatures.from_schema(
    schema,
    max_sequence_length=20,
    continuous_projection=64,
    masking="mlm",
    d_output=100,
)

transformer_config = tr.XLNetConfig.build(
    d_model=64, n_head=4, n_layer=2, total_seq_length=20
)

body = tr.SequentialBlock(
    inputs, tr.MLPBlock([64]), tr.TransformerBlock(transformer_config, masking=inputs.masking)
)

metrics = [
    NDCGAt(top_ks=[20, 40], labels_onehot=True),
    RecallAt(top_ks=[20, 40], labels_onehot=True),
]

head = tr.Head(
    body,
    tr.NextItemPredictionTask(weight_tying=True, metrics=metrics),
    inputs=inputs,
)

model = tr.Model(head)
print(model)

## Training Arguments & Trainer

In [ ]:
from transformers4rec.config.trainer import T4RecTrainingArguments
from transformers4rec.torch import Trainer

per_device_train_batch_size = int(os.environ.get("per_device_train_batch_size", '128'))
per_device_eval_batch_size = int(os.environ.get("per_device_eval_batch_size", '32'))

train_args = T4RecTrainingArguments(
    data_loader_engine='merlin',
    dataloader_drop_last=True,
    gradient_accumulation_steps=1,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    output_dir="./tmp",
    learning_rate=0.0005,
    lr_scheduler_type='cosine',
    learning_rate_num_cosine_cycles_by_epoch=1.5,
    num_train_epochs=5,
    max_sequence_length=20,
    report_to=[],
    logging_steps=50,
    no_cuda=False,
)

trainer = Trainer(
    model=model,
    args=train_args,
    schema=schema,
    compute_metrics=True,
)

## Daily Fine-Tuning: Train & Evaluate over 7 Days

In [ ]:
start_window_index = int(os.environ.get("start_window_index", '1'))
final_window_index = int(os.environ.get("final_window_index", '8'))

for time_index in range(start_window_index, final_window_index):
    time_index_train = time_index
    time_index_eval = time_index + 1
    train_paths = glob.glob(os.path.join(OUTPUT_DIR, f"{time_index_train}/train.parquet"))
    eval_paths = glob.glob(os.path.join(OUTPUT_DIR, f"{time_index_eval}/valid.parquet"))
    print(train_paths)

    print('*' * 20)
    print("Launch training for day %s are:" % time_index)
    print('*' * 20 + '\n')
    trainer.train_dataset_or_path = train_paths
    trainer.reset_lr_scheduler()
    trainer.train()
    trainer.state.global_step += 1
    print('finished')

    trainer.eval_dataset_or_path = eval_paths
    train_metrics = trainer.evaluate(metric_key_prefix='eval')
    print('*' * 20)
    print("Eval results for day %s are:\t" % time_index_eval)
    print('\n' + '*' * 20 + '\n')
    for key in sorted(train_metrics.keys()):
        print(" %s = %s" % (key, str(train_metrics[key])))
    wipe_memory()

## Re-compute Eval Metrics & Save Model

In [ ]:
eval_data_paths = glob.glob(os.path.join(OUTPUT_DIR, f"{time_index_eval}/valid.parquet"))
eval_metrics = trainer.evaluate(eval_dataset=eval_data_paths, metric_key_prefix='eval')
for key in sorted(eval_metrics.keys()):
    print("  %s = %s" % (key, str(eval_metrics[key])))

model_path = os.environ.get("OUTPUT_DIR", f"{INPUT_DATA_DIR}/saved_model")
model.save(model_path)
print(f"\nModel saved to {model_path}")